# Enrichment Scaling Experiment

**Key design change:** Mode 1 now evaluates **ordering direction recovery**, not
exact deleted-edge identification. For each newly-incomparable pair (a,b), the
label is:
- y=1 if a→b was reachable in the original DAG (adding a→b restores correct ordering)
- y=0 if b→a was reachable instead (adding a→b would reverse the ordering)

This gives ~50% positive rate per plan (one direction correct, one wrong) and
tests what the probe actually learns: the direction of temporal ordering.

**All other design choices unchanged:** integer node IDs throughout, dev-locked
thresholds, probe and LLM evaluated on identical candidate sets and labels.

**Conditions:** M1 del 20/40/60% · M2a same-depth n=1/2/3 · M2b cross-branch n=1/2/3 · Combined

**Metrics:** edge F1 · PRR · GEDR · TCA

**Required:** `dev.jsonl`, `test.jsonl`, `probe_manifest.json`, probe PKLs


In [ ]:
import subprocess, glob
subprocess.run(['pip','install','-q','transformers','accelerate',
                'scikit-learn','tqdm','matplotlib'], check=True)
import json as _json
from pathlib import Path
CONTENT = Path('/content')

for f in ['dev.jsonl','test.jsonl','probe_manifest.json']:
    assert (CONTENT/f).exists(), f'Missing: {f}'

manifest = _json.load(open(CONTENT/'probe_manifest.json'))
print(f'Manifest: {manifest}')
for k in ['probe1_file','probe2_file']:
    assert (CONTENT/manifest[k]).exists(), f'Missing: {manifest[k]}'
print('✓ All files present')


In [ ]:
import warnings, random, math
import numpy as np, pandas as pd

cfg = {
    'model_name':      manifest['model_name'],
    'deletion_rates':  [0.20, 0.40, 0.60],
    'spurious_levels': [1, 2, 3],
    'combined_del':    0.40,
    'combined_spur':   2,
    'thresholds':      [0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70],
    'chat_template':   True,
    'seed':            42,
    'ref_del_rate':    0.40,
    'ref_spur_level':  2,
    'n_corruption_seeds': 1,
}
SEED = cfg['seed']
print('Config ready')


In [ ]:
import json as _json
from collections import defaultdict, deque

def load_jsonl_full(path):
    plans, seen = {}, set()
    with open(path, encoding='utf-8-sig') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line: continue
            d = _json.loads(line)
            goal = d.get('scenario') or d.get('goal') or d.get('id','')
            if goal in seen: goal = f'{goal}__dup{i}'
            seen.add(goal); plans[goal] = d
    return plans

def parse_plan(d):
    raw_steps = d.get('events') or d.get('steps', {})
    steps = {int(k): v.strip() for k, v in raw_steps.items()}
    sentinels = {k for k, v in steps.items()
                 if v.upper() in ('START', 'END', 'NONE')}
    real = [k for k in steps if k not in sentinels]
    adj = defaultdict(list)
    id_edges = []
    raw_edges = d.get('gold_edges_for_prediction') or d.get('edges', [])
    for e in raw_edges:
        if isinstance(e, str):
            parts = e.split('->')
            if len(parts) != 2: continue
            a, b = int(parts[0]), int(parts[1])
        else:
            a, b = int(e[0]), int(e[1])
        if a in sentinels or b in sentinels: continue
        if a in steps and b in steps:
            adj[a].append(b); id_edges.append((a, b))
    return steps, id_edges, adj, real

def _reach(s, adj):
    v, q = set(), [s]
    while q:
        n = q.pop()
        for nb in adj.get(n, []):
            if nb not in v: v.add(nb); q.append(nb)
    return v

def _reach_all(real, adj):
    return {n: _reach(n, adj) for n in real}

def depths_map(real, adj):
    in_d = {s:0 for s in real}
    for n in real:
        for nb in adj.get(n,[]): in_d[nb] = in_d.get(nb,0)+1
    d = {s:0 for s in real}
    q = deque([s for s in real if in_d.get(s,0)==0])
    while q:
        n = q.popleft()
        for nb in adj.get(n,[]):
            d[nb] = max(d[nb], d[n]+1); in_d[nb] -= 1
            if in_d[nb]==0: q.append(nb)
    return d

def incompat_by_type(real, adj):
    r  = _reach_all(real, adj)
    dp = depths_map(real, adj)
    same, cross = [], []
    for i in range(len(real)):
        for j in range(i+1, len(real)):
            a, b = real[i], real[j]
            if b not in r[a] and a not in r[b]:
                (same if dp.get(a)==dp.get(b) else cross).append((a,b))
    return same, cross

def newly_incomparable_with_labels(real, adj_orig, adj_corrupt):
    """
    Returns list of (a, b, y) where:
      - (a,b) is incomparable in corrupted DAG but was reachable in original
      - y=1 if a could reach b in original (adding a→b restores correct ordering)
      - y=0 if b could reach a in original (adding a→b would reverse ordering)
    
    This gives ~50% positive rate: for each undirected pair that lost
    reachability, one direction is correct and the other is wrong.
    """
    r_orig    = _reach_all(real, adj_orig)
    r_corrupt = _reach_all(real, adj_corrupt)
    labeled = []
    for a in real:
        for b in real:
            if a == b: continue
            # Must be incomparable in corrupted DAG
            if b in r_corrupt[a] or a in r_corrupt[b]: continue
            # Must have been reachable in original in at least one direction
            if b in r_orig[a]:
                labeled.append((a, b, 1))  # a→b was correct ordering
            elif a in r_orig[b]:
                labeled.append((a, b, 0))  # b→a was correct, so a→b is wrong
    return labeled

def rebuild_adj(id_edges, real):
    adj = defaultdict(list)
    for a, b in id_edges: adj[a].append(b)
    return adj

def has_alt_path(id_edges, a, b):
    adj = defaultdict(set)
    for x, y in id_edges:
        if not (x==a and y==b): adj[x].add(y)
    v, q = set(), [a]
    while q:
        n = q.pop()
        if n == b: return True
        for nb in adj.get(n, set()):
            if nb not in v: v.add(nb); q.append(nb)
    return False

# ── Metrics ───────────────────────────────────────────────────────────────────
def edge_f1(tp, fp, fn):
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    return 2*p*r/(p+r) if p+r else 0.0, p, r

def calc_gedr(true_set, corrupt_set, output_set):
    d_in  = len(corrupt_set ^ true_set)
    d_out = len(output_set  ^ true_set)
    return (d_in - d_out) / d_in if d_in else 1.0

def calc_prr(true_set, output_set):
    return int(true_set == output_set)

def transitive_closure(real, adj):
    r = _reach_all(real, adj)
    return {(a, b) for a in real for b in r[a]}

def calc_tca(true_edges, output_edges, real):
    adj_t = rebuild_adj(list(true_edges), real)
    adj_o = rebuild_adj(list(output_edges), real)
    tc_t  = transitive_closure(real, adj_t)
    tc_o  = transitive_closure(real, adj_o)
    total = len(real) * (len(real)-1) if len(real)>1 else 1
    agree = total - len(tc_t ^ tc_o)
    return agree / total

print('Utils ready')


In [ ]:
import torch, pickle
from transformers import AutoTokenizer, AutoModelForCausalLM

def wrap(raw):
    return f'[INST] {raw.strip()} [/INST]' if cfg['chat_template'] else raw

def tp1_ordering(goal, a, b):
    return wrap(
        f'You are judging a temporal dependency between two actions in a task.\n'
        f'Task: {goal}\nAction A: {a}\nAction B: {b}\n'
        f'Question: Must Action A happen before Action B? '
        f'Answer yes or no.\nAnswer:')

def _unique_ids(variants, tok):
    ids = set()
    for t in variants:
        enc = tok.encode(t, add_special_tokens=False)
        if len(enc)==1: ids.add(enc[0])
    return sorted(ids)

print(f'Loading {cfg["model_name"]} ...')
tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])
model = AutoModelForCausalLM.from_pretrained(
    cfg['model_name'], torch_dtype=torch.float16,
    device_map='auto', output_hidden_states=True)
model.eval()
YES_IDS = _unique_ids([' yes','yes','Yes',' Yes'], tokenizer)
NO_IDS  = _unique_ids([' no', 'no', 'No', ' No'], tokenizer)

def load_probe_from_manifest(key):
    data  = pickle.load(open(CONTENT / manifest[f'{key}_file'], 'rb'))
    layer = data['layer'] if isinstance(data, dict) else manifest[f'{key}_layer']
    probe = data['probe'] if isinstance(data, dict) else data
    return probe, layer

probe1, p1_layer = load_probe_from_manifest('probe1')
probe2, p2_layer = load_probe_from_manifest('probe2')
print(f'Probe 1 at layer {p1_layer}  |  Probe 2 at layer {p2_layer}')

@torch.no_grad()
def score(goal, steps, a_id, b_id):
    """Single forward pass → probe1 score, probe2 spurious score, p_yes.
    Used identically for both probe and LLM evaluation."""
    prompt = tp1_ordering(goal, steps[a_id], steps[b_id])
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out    = model(**inputs, output_hidden_states=True)
    h1 = out.hidden_states[p1_layer][0,-1].float().cpu().numpy().reshape(1,-1)
    h2 = out.hidden_states[p2_layer][0,-1].float().cpu().numpy().reshape(1,-1)
    probs = torch.softmax(out.logits[0,-1].float(), dim=-1).cpu()
    p_yes = sum(probs[i].item() for i in YES_IDS)
    p_no  = sum(probs[i].item() for i in NO_IDS)
    denom = p_yes+p_no if p_yes+p_no>0 else 1.0
    return float(probe1.predict_proba(h1)[0,1]), \
           float(probe2.predict_proba(h2)[0,0]), \
           p_yes/denom

print('Scoring ready (same call used for probe + LLM)')


In [ ]:
from tqdm.auto import tqdm

dev_all  = load_jsonl_full(CONTENT/'dev.jsonl')
test_all = load_jsonl_full(CONTENT/'test.jsonl')

def filter_multi(plans):
    out = {}
    for goal, d in plans.items():
        steps, id_edges, adj, real = parse_plan(d)
        same, cross = incompat_by_type(real, adj)
        if same or cross:
            out[goal] = {'steps':steps, 'id_edges':id_edges,
                         'adj':adj, 'real':real,
                         'same':same, 'cross':cross}
    return out

dev_multi  = filter_multi(dev_all)
test_multi = filter_multi(test_all)
print(f'Multi-ordering — dev: {len(dev_multi)}  test: {len(test_multi)}')


In [ ]:
def delete_edges(id_edges, rate, rng_):
    hard = [(a,b) for a,b in id_edges if not has_alt_path(id_edges, a, b)]
    if not hard: return id_edges, []
    n = max(1, math.ceil(rate * len(hard)))
    deleted = set(rng_.sample(hard, min(n, len(hard))))
    return [e for e in id_edges if e not in deleted], list(deleted)

def add_spurious(pairs, n_spur, rng_):
    selected = rng_.sample(pairs, min(n_spur, len(pairs)))
    return [(a,b) if rng_.random() < 0.5 else (b,a)
            for a,b in selected]

print('Corruption helpers ready')


---
## Dev pass — threshold selection

Mode 1 candidates: newly-incomparable pairs labeled by **reachability direction**
(y=1 if a→b was reachable in original, y=0 if b→a was). ~50% positive rate.

Mode 2 candidates: all edges in corrupted DAG (unchanged).

Both probe and LLM see identical candidate pairs and labels.
One threshold per mode, held fixed across all severities.


In [ ]:
rng_dev = random.Random(SEED)
locked  = {}

def run_sweep(plans, mode, param, rng_):
    rows = []
    for goal, p in tqdm(plans.items(), desc=f'{mode} {param}', leave=False):
        true_e = p['id_edges']
        steps  = p['steps']
        real   = p['real']

        if mode == 'M1':
            corrupt, deleted = delete_edges(true_e, param, rng_)
            if not deleted: continue
            adj_c = rebuild_adj(corrupt, real)
            # Direction-labeled newly-incomparable pairs
            candidates = newly_incomparable_with_labels(real, p['adj'], adj_c)
            for a, b, y in candidates:
                s1, _, py = score(goal, steps, a, b)
                # Both probe and LLM answer the same question:
                # "should A come before B?" → high score = yes
                rows.append({'y':y, 's':s1, 'py':py})
        else:
            pairs = p['same'] if mode=='M2a' else p['cross']
            if not pairs: continue
            added   = add_spurious(pairs, param, rng_)
            if not added: continue
            add_set = set(map(tuple, added))
            corrupt = true_e + added
            for a, b in corrupt:
                _, s2, py = score(goal, steps, a, b)
                rows.append({'y':int((a,b) in add_set), 's':s2, 'py':py})
    return pd.DataFrame(rows)

ref_conditions = [
    ('M1',  cfg['ref_del_rate']),
    ('M2a', cfg['ref_spur_level']),
    ('M2b', cfg['ref_spur_level']),
]

for mode, param in ref_conditions:
    df = run_sweep(dev_multi, mode, param, rng_dev)
    if df.empty:
        print(f'  WARNING: {mode} — no data'); continue
    best_pf, best_pt = -1, 0.50
    best_lf, best_lt = -1, 0.50
    for t in cfg['thresholds']:
        # Probe: high score → predict positive (same for M1 and M2)
        pp = (df.s > t).astype(int)
        # LLM: M1 high p_yes = "yes A before B" → positive
        #      M2 low p_yes = "not confident about ordering" → spurious
        lp = (df.py > t if mode=='M1' else df.py < t).astype(int)
        for sys, preds, bfr, btr in [('probe',pp,best_pf,best_pt),
                                      ('llm',  lp,best_lf,best_lt)]:
            tp = int(((preds==1)&(df.y==1)).sum())
            fp = int(((preds==1)&(df.y==0)).sum())
            fn = int(((preds==0)&(df.y==1)).sum())
            f1v,_,_ = edge_f1(tp, fp, fn)
            if sys=='probe' and f1v > best_pf:
                best_pf, best_pt = f1v, t
            if sys=='llm' and f1v > best_lf:
                best_lf, best_lt = f1v, t
    locked[(mode,'probe')] = best_pt
    locked[(mode,'llm')]   = best_lt
    pos_rate = df.y.mean() * 100
    print(f'  DEV {mode} ref={param}: probe F1={best_pf:.3f} t={best_pt} '
          f'| llm F1={best_lf:.3f} t={best_lt} '
          f'| pos_rate={pos_rate:.1f}%  n={len(df)}')

print(f'\n✓ Locked: {locked}')


---
## Test pass — final evaluation

Locked thresholds applied at all severity levels.

Mode 1: probe and LLM both answer "should A come before B?" on the same
newly-incomparable pairs with direction labels. High score → predict yes.

Mode 2: probe scores P(spurious), LLM uses low p_yes as spurious signal.
Both see the same edges with the same labels.


In [ ]:
test_results = []

def eval_test(plans, mode, param, base_seed):
    t_probe = locked.get((mode,'probe'), 0.50)
    t_llm   = locked.get((mode,'llm'),   0.50)
    all_pairs, all_plans = [], []

    for si in range(cfg['n_corruption_seeds']):
        rng_ = random.Random(base_seed + si)
        for goal, p in plans.items():
            true_e   = p['id_edges']
            true_set = set(map(tuple, true_e))
            steps    = p['steps']
            real     = p['real']

            if mode == 'M1':
                corrupt, deleted = delete_edges(true_e, param, rng_)
                if not deleted: continue
                adj_c = rebuild_adj(corrupt, real)
                candidates = newly_incomparable_with_labels(real, p['adj'], adj_c)
                prop_p, prop_l = set(), set()
                for a, b, y in candidates:
                    s1, _, py = score(goal, steps, a, b)
                    all_pairs.append({'y':y,
                        'pred_p':int(s1>t_probe), 'pred_l':int(py>t_llm)})
                    if s1 > t_probe: prop_p.add((a,b))
                    if py > t_llm:   prop_l.add((a,b))
                out_p = set(map(tuple,corrupt)) | prop_p
                out_l = set(map(tuple,corrupt)) | prop_l

            else:
                pairs = p['same'] if mode=='M2a' else p['cross']
                if not pairs: continue
                added = add_spurious(pairs, param, rng_)
                if not added: continue
                add_set = set(map(tuple, added))
                corrupt = true_e + added
                rem_p, rem_l = set(), set()
                for a, b in corrupt:
                    _, s2, py = score(goal, steps, a, b)
                    y = int((a,b) in add_set)
                    all_pairs.append({'y':y,
                        'pred_p':int(s2>t_probe), 'pred_l':int(py<t_llm)})
                    if s2 > t_probe: rem_p.add((a,b))
                    if py < t_llm:   rem_l.add((a,b))
                out_p = set(e for e in map(tuple,corrupt) if e not in rem_p)
                out_l = set(e for e in map(tuple,corrupt) if e not in rem_l)

            cor_set = set(map(tuple, corrupt))
            all_plans.append({
                'prr_p':calc_prr(true_set, out_p), 'prr_l':calc_prr(true_set, out_l),
                'gedr_p':calc_gedr(true_set, cor_set, out_p),
                'gedr_l':calc_gedr(true_set, cor_set, out_l),
                'tca_p':calc_tca(true_set, out_p, real),
                'tca_l':calc_tca(true_set, out_l, real),
            })

    if not all_pairs: return
    df = pd.DataFrame(all_pairs)
    pm = pd.DataFrame(all_plans)
    for sys, pc, prr_c, gedr_c, tca_c in [
        ('probe','pred_p','prr_p','gedr_p','tca_p'),
        ('llm',  'pred_l','prr_l','gedr_l','tca_l')]:
        tp = int(((df[pc]==1)&(df.y==1)).sum())
        fp = int(((df[pc]==1)&(df.y==0)).sum())
        fn = int(((df[pc]==0)&(df.y==1)).sum())
        f1v, prec, rec = edge_f1(tp, fp, fn)
        test_results.append({
            'mode':mode,'param':param,'system':sys,
            'f1':round(f1v,4),'precision':round(prec,4),'recall':round(rec,4),
            'tp':tp,'fp':fp,'fn':fn,
            'prr_mean':round(float(pm[prr_c].mean()),4),
            'gedr_mean':round(float(pm[gedr_c].mean()),4),
            'tca_mean':round(float(pm[tca_c].mean()),4),
            'n_plans':len(pm),
            'pos_rate':round(float(df.y.mean())*100,1),
        })

conditions = (
    [('M1',  r) for r in cfg['deletion_rates']] +
    [('M2a', n) for n in cfg['spurious_levels']] +
    [('M2b', n) for n in cfg['spurious_levels']])

for mode, param in conditions:
    if (mode,'probe') not in locked: continue
    print(f'Running {mode} {param}...')
    eval_test(test_multi, mode, param, SEED+10)

res_df = pd.DataFrame(test_results)
print('\n=== TEST RESULTS ===')
for sys in ['probe','llm']:
    print(f'\n  {sys.upper()}:')
    sub = res_df[res_df.system==sys]
    print(sub[['mode','param','f1','precision','recall',
               'prr_mean','gedr_mean','tca_mean',
               'pos_rate','n_plans']].to_string(index=False))


In [ ]:
print('=== COMBINED (40% del + 2 same-depth spur) ===')
rng_c = random.Random(SEED+20)
comb_pm = []

t1_p = locked.get(('M1','probe'),  0.50)
t2_p = locked.get(('M2a','probe'), 0.50)
t1_l = locked.get(('M1','llm'),    0.50)
t2_l = locked.get(('M2a','llm'),   0.50)

for goal, p in tqdm(test_multi.items(), desc='Combined'):
    true_e   = p['id_edges']
    true_set = set(map(tuple, true_e))
    steps    = p['steps']
    real     = p['real']

    corrupt, deleted = delete_edges(true_e, cfg['combined_del'], rng_c)
    if not deleted: continue
    adj_d = rebuild_adj(corrupt, real)
    same_r, _ = incompat_by_type(real, adj_d)
    if not same_r: continue
    added   = add_spurious(same_r, cfg['combined_spur'], rng_c)
    corrupt = corrupt + added

    adj_c = rebuild_adj(corrupt, real)
    # Mode 1: direction-labeled newly-incomparable
    candidates = newly_incomparable_with_labels(real, p['adj'], adj_c)

    prop_add_p, prop_add_l = set(), set()
    for a, b, y in candidates:
        s1, _, py = score(goal, steps, a, b)
        if s1 > t1_p: prop_add_p.add((a,b))
        if py > t1_l: prop_add_l.add((a,b))

    # Mode 2: removal proposals
    prop_rem_p, prop_rem_l = set(), set()
    for a, b in corrupt:
        _, s2, py = score(goal, steps, a, b)
        if s2 > t2_p: prop_rem_p.add((a,b))
        if py < t2_l: prop_rem_l.add((a,b))

    cor_set  = set(map(tuple, corrupt))
    output_p = (cor_set | prop_add_p) - prop_rem_p
    output_l = (cor_set | prop_add_l) - prop_rem_l

    comb_pm.append({
        'prr_p':calc_prr(true_set, output_p),   'prr_l':calc_prr(true_set, output_l),
        'gedr_p':calc_gedr(true_set, cor_set, output_p),
        'gedr_l':calc_gedr(true_set, cor_set, output_l),
        'tca_p':calc_tca(true_set, output_p, real),
        'tca_l':calc_tca(true_set, output_l, real),
    })

comb = pd.DataFrame(comb_pm)
if comb.empty:
    print('No eligible plans.')
else:
    print(f'Combined ({len(comb)} plans):')
    print(f'  Probe: PRR={comb.prr_p.mean():.3f}  '
          f'GEDR={comb.gedr_p.mean():.3f}  TCA={comb.tca_p.mean():.3f}')
    print(f'  LLM:   PRR={comb.prr_l.mean():.3f}  '
          f'GEDR={comb.gedr_l.mean():.3f}  TCA={comb.tca_l.mean():.3f}')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
rates  = [int(r*100) for r in cfg['deletion_rates']]
levels = cfg['spurious_levels']

def _get(mode, param, sys, field):
    r = res_df[(res_df['mode']==mode)&(res_df.param==param)&(res_df.system==sys)]
    return float(r[field].values[0]) if len(r) else 0.0

ax = axes[0,0]
for sys, col, ls in [('probe','#3C3489','-'), ('llm','#888780','--')]:
    ax.plot(rates, [_get('M1',r,sys,'f1') for r in cfg['deletion_rates']],
            f'o{ls}', color=col, lw=2, ms=6, label=sys)
ax.set(xlabel='Hard edges deleted (%)', ylabel='Edge F1',
       title='Mode 1: F1 vs deletion rate (direction recovery)'); ax.legend(); ax.set_ylim(0,1)

ax = axes[0,1]
for field, lab, col, ls in [('prr_mean','PRR','#185FA5','-'),
                             ('gedr_mean','GEDR','#1D9E75','--'),
                             ('tca_mean','TCA','#E24B4A',':')]:
    ax.plot(rates, [_get('M1',r,'probe',field) for r in cfg['deletion_rates']],
            f'o{ls}', color=col, lw=2, ms=5, label=lab)
ax.set(xlabel='Hard edges deleted (%)', ylabel='Score',
       title='Mode 1: PRR / GEDR / TCA (probe)'); ax.legend()
gv = [_get('M1',r,'probe','gedr_mean') for r in cfg['deletion_rates']]
ax.set_ylim(min(-0.1, min(gv)-0.05) if gv else -0.1, 1.05)

ax = axes[1,0]
for mode, col in [('M2a','#3C3489'), ('M2b','#185FA5')]:
    ax.plot(levels, [_get(mode,n,'probe','f1') for n in levels],
            'o-', color=col, lw=2, ms=6, label=f'{mode} probe')
    ax.plot(levels, [_get(mode,n,'llm','f1') for n in levels],
            's--', color=col, lw=1, ms=5, alpha=0.5, label=f'{mode} llm')
ax.set(xlabel='Spurious edges added', ylabel='Edge F1',
       title='Mode 2: F1 vs spurious level'); ax.legend(fontsize=8); ax.set_ylim(0,1)

ax = axes[1,1]
for mode, col, ls in [('M2a','#3C3489','-'), ('M2b','#185FA5','--')]:
    for field, mk in [('prr_mean','o'), ('gedr_mean','s'), ('tca_mean','^')]:
        lab = f'{mode} {field.split("_")[0].upper()}'
        ax.plot(levels, [_get(mode,n,'probe',field) for n in levels],
                f'{mk}{ls}', color=col, lw=1.5, ms=5, alpha=0.7, label=lab)
ax.set(xlabel='Spurious edges added', ylabel='Score',
       title='Mode 2: PRR / GEDR / TCA (probe)'); ax.legend(fontsize=7)
gv2 = [_get(m,n,'probe','gedr_mean') for m in ['M2a','M2b'] for n in levels]
ax.set_ylim(min(-0.1, min(gv2)-0.05) if gv2 else -0.1, 1.05)

plt.tight_layout()
plt.savefig(CONTENT/'enrichment_scaling.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
res_df.to_csv(CONTENT/'enrichment_results.csv', index=False)
pd.DataFrame([{'mode':m,'system':s,'threshold':t}
               for (m,s),t in locked.items()])\
  .to_csv(CONTENT/'locked_thresholds.csv', index=False)

if not comb.empty:
    comb_label = f'{int(cfg["combined_del"]*100)}pct+{cfg["combined_spur"]}spur'
    comb_rows = []
    for sys, prr_c, gedr_c, tca_c in [
        ('probe','prr_p','gedr_p','tca_p'),
        ('llm',  'prr_l','gedr_l','tca_l')]:
        comb_rows.append({'mode':'combined','param':comb_label,'system':sys,
                          'prr_mean':round(float(comb[prr_c].mean()),4),
                          'gedr_mean':round(float(comb[gedr_c].mean()),4),
                          'tca_mean':round(float(comb[tca_c].mean()),4),
                          'n_plans':len(comb)})
    res_df = pd.concat([res_df, pd.DataFrame(comb_rows)], ignore_index=True)
    res_df.to_csv(CONTENT/'enrichment_results.csv', index=False)

print('Saved: enrichment_results.csv  locked_thresholds.csv  enrichment_scaling.png')
print('\n=== FINAL SUMMARY ===')
for _, row in res_df[res_df.system=='probe'].iterrows():
    f1s = f'F1={row["f1"]:.4f}' if pd.notna(row.get('f1')) else ''
    print(f'  {row["mode"]} {row["param"]}: {f1s}  '
          f'PRR={row.get("prr_mean","")!s:>6}  '
          f'GEDR={row.get("gedr_mean","")!s:>7}  '
          f'TCA={row.get("tca_mean","")!s:>6}')
